# Advanced NLP Text Features

Computes lightweight text statistics for title and bullet points. All features can run on a MacBook Air (no transformers/BERT).

**Run after:** `03_tfidf_features.ipynb` (which saves the enriched CSV)

**New features (~15):**
- Title: char count, word count, avg word length, unique word ratio, reading ease, grade level, separator count, has_brand, has_size_spec, has_color_spec
- Bullets: count, avg length, total word count, keyword density

In [1]:
import re
import warnings
import numpy as np
import pandas as pd
import textstat

warnings.filterwarnings('ignore')

DATA_PATH = "../../data/products_with_image_feats.csv"

print("Loading dataset...")
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} products, {len(df.columns)} columns")

# Ensure text columns exist (from previous notebook)
if 'text_title_clean' not in df.columns:
    print("text_title_clean not found, rebuilding from raw columns...")
    def clean_text(text):
        if pd.isna(text) or not isinstance(text, str) or text.strip() == '':
            return ''
        text = text.lower()
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    
    if 'sd_title' in df.columns:
        df['text_title'] = df['sd_title'].fillna(df.get('title', '')).fillna('')
    else:
        df['text_title'] = df.get('title', '')
    df['text_title_clean'] = df['text_title'].apply(clean_text)
    
    if 'sd_feature_bullets_text' in df.columns:
        df['text_bullets'] = df['sd_feature_bullets_text'].fillna('')
    else:
        df['text_bullets'] = ''
    df['text_bullets_clean'] = df['text_bullets'].apply(clean_text)

print("Text columns ready.")

Loading dataset...
Loaded 36879 products, 7756 columns
Text columns ready.


## 1. Title Text Statistics

In [2]:
# Use the raw title (with original casing and punctuation) for some features
# and the clean version for others
raw_title_col = 'text_title' if 'text_title' in df.columns else 'sd_title' if 'sd_title' in df.columns else 'title'
raw_title = df[raw_title_col].fillna('') if raw_title_col in df.columns else pd.Series([''] * len(df))

clean_title = df['text_title_clean'].fillna('')

# Basic length features
df['title_char_count'] = clean_title.str.len()
df['title_word_count'] = clean_title.apply(lambda x: len(x.split()) if x else 0)
df['title_avg_word_length'] = clean_title.apply(
    lambda x: np.mean([len(w) for w in x.split()]) if x and len(x.split()) > 0 else 0
)

# Vocabulary richness
df['title_unique_word_ratio'] = clean_title.apply(
    lambda x: len(set(x.split())) / max(len(x.split()), 1) if x else 0
)

# Readability scores (textstat)
print("Computing readability scores (this may take a minute)...")
df['title_flesch_reading_ease'] = clean_title.apply(
    lambda x: textstat.flesch_reading_ease(x) if x and len(x.split()) >= 3 else 0
)
df['title_flesch_kincaid_grade'] = clean_title.apply(
    lambda x: textstat.flesch_kincaid_grade(x) if x and len(x.split()) >= 3 else 0
)

# Title structure features (using raw title to detect separators)
df['title_separator_count'] = raw_title.apply(
    lambda x: x.count('|') + x.count(' - ') + x.count(',') if isinstance(x, str) else 0
)

# Brand detection
brand_col = 'brand' if 'brand' in df.columns else None
if brand_col:
    df['title_has_brand'] = df.apply(
        lambda row: 1 if (isinstance(row.get(brand_col), str) and 
                          isinstance(row.get(raw_title_col), str) and
                          row[brand_col].lower() in row[raw_title_col].lower()) else 0,
        axis=1
    )
else:
    df['title_has_brand'] = 0

# Size/dimension detection
SIZE_PATTERN = re.compile(
    r'\b\d+\s*(?:oz|inch|in|cm|mm|ml|liter|litre|lb|lbs|kg|ft|feet|gallon|gal|qt|quart|pack|pc|pcs|count|ct)\b',
    re.IGNORECASE
)
df['title_has_size_spec'] = raw_title.apply(
    lambda x: 1 if isinstance(x, str) and SIZE_PATTERN.search(x) else 0
)

# Color detection
COLOR_PATTERN = re.compile(
    r'\b(?:black|white|red|blue|green|yellow|pink|purple|orange|gold|silver|grey|gray|brown|navy|teal|beige|ivory)\b',
    re.IGNORECASE
)
df['title_has_color_spec'] = raw_title.apply(
    lambda x: 1 if isinstance(x, str) and COLOR_PATTERN.search(x) else 0
)

# Summary
title_features = [c for c in df.columns if c.startswith('title_') and c not in ['title_tfidf_pca_' + f'{i:04d}' for i in range(50)]]
title_features = [c for c in title_features if not c.startswith('title_tfidf_')]
print(f"\nCreated {len(title_features)} title features:")
for feat in title_features:
    print(f"  {feat}: mean={df[feat].mean():.2f}, std={df[feat].std():.2f}")

Computing readability scores (this may take a minute)...

Created 10 title features:
  title_char_count: mean=137.86, std=44.75
  title_word_count: mean=22.87, std=7.61
  title_avg_word_length: mean=5.12, std=0.65
  title_unique_word_ratio: mean=0.91, std=0.08
  title_flesch_reading_ease: mean=48.73, std=18.15
  title_flesch_kincaid_grade: mean=12.13, std=3.59
  title_separator_count: mean=3.12, std=2.52
  title_has_brand: mean=0.00, std=0.00
  title_has_size_spec: mean=0.31, std=0.46
  title_has_color_spec: mean=0.34, std=0.47


## 2. Bullet Point Features

In [3]:
raw_bullets = df['sd_feature_bullets_text'].fillna('') if 'sd_feature_bullets_text' in df.columns else pd.Series([''] * len(df))
clean_bullets = df['text_bullets_clean'].fillna('')

# Split bullets by newlines or bullet markers
def count_bullets(text):
    if not isinstance(text, str) or not text.strip():
        return 0
    # Split on newlines, bullet chars, or pipes
    parts = re.split(r'[\n\r|]+', text)
    return len([p for p in parts if p.strip()])

df['bullets_count'] = raw_bullets.apply(count_bullets)

# Average bullet length
def avg_bullet_length(text):
    if not isinstance(text, str) or not text.strip():
        return 0
    parts = re.split(r'[\n\r|]+', text)
    parts = [p.strip() for p in parts if p.strip()]
    if not parts:
        return 0
    return np.mean([len(p) for p in parts])

df['bullets_avg_length'] = raw_bullets.apply(avg_bullet_length)

# Total word count in bullets
df['bullets_total_word_count'] = clean_bullets.apply(
    lambda x: len(x.split()) if x else 0
)

# Has bullets flag
df['has_bullets'] = (df['bullets_count'] > 0).astype(int)

# Bullet keyword density (ratio of meaningful words vs total)
FILLER_WORDS = set(['the', 'a', 'an', 'and', 'or', 'but', 'is', 'are', 'was', 'were', 
                     'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'from', 'as',
                     'this', 'that', 'it', 'its', 'our', 'your', 'you', 'we', 'they'])

def keyword_density(text):
    if not text or not text.strip():
        return 0
    words = text.split()
    if not words:
        return 0
    meaningful = [w for w in words if w not in FILLER_WORDS and len(w) > 2]
    return len(meaningful) / len(words)

df['bullets_keyword_density'] = clean_bullets.apply(keyword_density)

# Summary
bullet_features = ['bullets_count', 'bullets_avg_length', 'bullets_total_word_count', 
                    'has_bullets', 'bullets_keyword_density']
print(f"Created {len(bullet_features)} bullet features:")
for feat in bullet_features:
    print(f"  {feat}: mean={df[feat].mean():.2f}, std={df[feat].std():.2f}")

Created 5 bullet features:
  bullets_count: mean=0.00, std=0.00
  bullets_avg_length: mean=0.00, std=0.00
  bullets_total_word_count: mean=0.00, std=0.00
  has_bullets: mean=0.00, std=0.00
  bullets_keyword_density: mean=0.00, std=0.00


## 3. Save Enriched Dataset

In [4]:
# List all new NLP features
all_nlp_features = [
    'title_char_count', 'title_word_count', 'title_avg_word_length',
    'title_unique_word_ratio', 'title_flesch_reading_ease', 'title_flesch_kincaid_grade',
    'title_separator_count', 'title_has_brand', 'title_has_size_spec', 'title_has_color_spec',
    'bullets_count', 'bullets_avg_length', 'bullets_total_word_count',
    'has_bullets', 'bullets_keyword_density',
]
print(f"Total NLP features: {len(all_nlp_features)}")
print(f"DataFrame shape: {df.shape}")

# Verify all features exist
for feat in all_nlp_features:
    assert feat in df.columns, f"Missing feature: {feat}"
print("All NLP features verified in dataframe.")

# Save
OUTPUT_CSV = "../../data/products_with_image_feats.csv"
print(f"\nSaving to {OUTPUT_CSV}...")
df.to_csv(OUTPUT_CSV, index=False)
print("Done! NLP text features added to dataset.")

Total NLP features: 15
DataFrame shape: (36879, 7771)
All NLP features verified in dataframe.

Saving to ../../data/products_with_image_feats.csv...
Done! NLP text features added to dataset.
